# 競艇予測モデル — Colab 実行ドライバ

**このノートブックは成果物ではありません。** 実装は `src/kyotei/` の再実行可能なスクリプトにあり、ここはそれを Colab から叩くだけの薄いドライバです（SPEC §5）。ロジックをこのノートに書かないでください。

## GPU とノートブックについて

**GPU は要りません。** SPEC §4 は PyTorch / NN を禁じて LightGBM (GBDT) を指定しています。GBDT はヒストグラム構築が CPU・メモリ帯域律速で、LightGBM に GPU ビルドはあるものの、この規模（数百万行・特徴量数十）ではまず速くなりません。ランタイムは **「CPU・ハイメモリ」** を選んでください。GPU を選ぶと待ち行列が長くなるだけ損です。

**ではなぜノートブックを使うのか。** ロジックをここに書くためではありません（SPEC §5 が Notebook を成果物にすることを禁じています。テストも書けなくなり、いま 340 あるテストが効かなくなります）。ノートブックが効くのは **Colab という実行環境そのもの**が持つ利点のためです:

| 課題 | Colab で解決 |
|---|---|
| 実オッズの取得に約15時間かかる | 中断・再開可能なので、セッションが切れても続きから |
| コンテナが揮発してデータが消える | `data/` を Drive に置いて永続化 |
| 学習の反復 | ハイメモリで全期間を一度に載せる |

つまり **ノートブックは長時間ジョブのホストとドライバ**であって、実装の置き場所ではありません。

Colab を使う本当の利点はこちらです:

| 課題 | Colab での解決 |
|---|---|
| 取得に約2.4時間かかる（サーバのTTFBが約10秒） | Drive に保存し、セッションが切れても再取得しない |
| コンテナが揮発してデータが消える | `data/` を Drive に置いて永続化 |
| 学習の反復 | メモリの大きいランタイムで全期間を一度に載せる |

## 手順

上から順に実行します。取得は**冪等**なので、途中で切れても再実行すれば続きから進みます。

## クイックスタート（実オッズの取得を再開する場合）

このリポジトリには**取得済みのオッズが同梱**されています（`data/odds/odds3t_2024.jsonl`）。
続きから再開するだけなら必要なのは4つだけです。

1. **1. Drive をマウント** → 2. **リポジトリと依存関係**（コミット済みオッズをDriveへ種付けしてからリンクします）
2. **3. テスト** を通す
3. **11-a. 2024年の番組表を取得**（366ファイル・約6分）— どのレースが存在するかを知るために必要
4. **11-c. 本番の取得** — セッションが切れたらこのセルを再実行するだけ

全期間の日次アーカイブ（手順4、約2.4時間）は**オッズ取得には不要**です。
後でバックテストする段階になってから回してください。

> **ランタイムは「CPU・ハイメモリ」を選んでください。** GPU は不要です
> （LightGBM の GBDT は CPU・メモリ帯域律速で、GPU では速くなりません）。


## 1. Drive をマウントしてデータを永続化する

`data/` を Drive 上に置き、リポジトリからシンボリックリンクします。これで**取得済みのLZHがセッションを越えて残ります**。

VS Code の Colab 拡張から実行する場合、`drive.mount` は拡張 v0.2.1 以降で使えます。認証が出ないときはコマンドパレットの `Colab: Mount Google Drive to Server...` を使ってください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
PERSIST = pathlib.Path('/content/drive/MyDrive/kyotei-predict')
(PERSIST / 'data').mkdir(parents=True, exist_ok=True)
(PERSIST / 'reports').mkdir(parents=True, exist_ok=True)
print('persisting to', PERSIST)

## 2. リポジトリと依存関係

`REPO_URL` を自分のリモートに置き換えてください。

In [ ]:
REPO_URL = 'https://github.com/Daccho/kyotei-predict.git'
BRANCH = 'main'

import os, pathlib, shutil, subprocess

REPO = pathlib.Path('/content/kyotei-predict')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'pull', '--ff-only'], check=False)

# Seed Drive from whatever data the repo carries BEFORE linking, then link.
#
# This order matters. data/odds/*.jsonl is committed because refetching a season
# costs ~15 hours, and it doubles as the resume ledger. Creating the symlink
# first would hide those committed records behind an empty Drive folder, and the
# backfill would start again from zero.
for name in ('data', 'reports'):
    local = REPO / name
    target = PERSIST / name
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.is_dir():
        for item in local.rglob('*'):
            if item.is_file():
                dest = target / item.relative_to(local)
                if not dest.exists():
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(item, dest)
                    print('seeded', dest.relative_to(target))
        shutil.rmtree(local)
    local.symlink_to(target)
    print(name, '->', local.resolve())

# Prove the committed odds survived the link.
jsonl = REPO / 'data' / 'odds' / 'odds3t_2024.jsonl'
if jsonl.exists():
    print('odds records carried over:', sum(1 for _ in jsonl.open()))

In [ ]:
# lhafile decodes LZH in pure Python, so no apt-get lhasa is needed.
!pip -q install 'polars>=1.0' 'lhafile>=0.3' 'lightgbm>=4.3' 'scikit-learn>=1.4' pyarrow matplotlib requests pytest

import importlib, pathlib, sys

# sys.path.insert() は存在しないパスでも黙って通るので、先に実体を確認する。
# ランタイムが張り替わると /content は消えるが、上のセルの出力は残ったままになる。
SRC = pathlib.Path('/content/kyotei-predict/src')
assert SRC.is_dir(), f'{SRC} がありません。先に 2. のセル (clone) を実行してください'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# 存在しないパスを一度探索すると sys.path_importer_cache に None が残り、
# あとからクローンしても同じカーネルでは import が通らない。捨ててから import する。
importlib.invalidate_caches()

import kyotei; print('kyotei ok:', kyotei.__file__)

## 3. テストを通す

**先にこれを実行してください。** リーク検証テスト（未来のレコードを改変しても過去の特徴量が変わらないこと）と、確率の合計が 1.0 になる検証がここに入っています。落ちている状態で先に進む意味はありません。

In [ ]:
!cd /content/kyotei-predict && python -m pytest tests/ -q --ignore=tests/test_schema.py

## 4. データ取得（冪等・全体1req/s上限）

オリジンの TTFB が約10秒あるため、待ち時間を12ワーカーで隠しつつ、**トークンバケットで全体を1req/s以下**に保っています。ワーカー数を増やしてもリクエスト頻度は上がりません（`--rate` が上限）。

全期間（2015–現在、約8450ファイル）で**約2.4時間**。Colab のセッション上限に当たったら、このセルをもう一度実行すれば取得済み分はスキップされて続きから進みます。

まず短い期間で試すなら `--end` を縮めてください。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.download --dry-run --start 2015-01-01 --end 2026-12-31 --fan

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.download --start 2015-01-01 --end 2026-12-31 --kind both --fan

## 5. パースして parquet 化（DB不要）

**年ごとのパース成功率**が出ます。特定の年だけ成功率が落ちていたらレイアウト変更のサインなので、先に原因を調べてください（SPEC §3.2）。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.export --start 2015-01-01 --end 2026-12-31

## 6. 特徴量（リーク防止）

過去成績はすべて「そのレースの発走前」のみから計算されます。累積和から現在行を引く形なので、未来を含む窓が構造的に存在しません。

In [ ]:
import polars as pl
from kyotei import features as ft

raw = pl.read_parquet('data/parquet/entries.parquet')
print('entries:', raw.height)

built = ft.build(raw)
assert built.height == raw.height, f'row count changed: {raw.height} -> {built.height}'
built.write_parquet('data/parquet/features.parquet')
print('features:', built.height, 'rows,', len(built.columns), 'columns')
print('morning feature count:', len(ft.MORNING_FEATURES))

## 7. 学習

`--feature-set morning` が**実際に賭けられる**設定です（番組表のみ）。`prerace` は直前情報を、`realised` は実際の進入コースを足しますが、どちらも締切前には手に入らないので**診断用の上限値**としてしか使えません。両方回して差を見ると、進入コースがどれだけ効いているかが分かります。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.model --feature-set morning --lane1-baseline --walk-forward

In [ ]:
# Diagnostic ceiling: how much the realised course is worth. NOT bettable.
!cd /content/kyotei-predict && python -m kyotei.model --feature-set realised \
    --model-out data/parquet/model_realised.txt

## 8. バックテスト（検証期間 2024）

閾値のチューニングは**必ず valid（2024年）で**行ってください。test（2025–2026）は最後に一度だけです。

回収率が 100% を超えたら、成功ではなくまずリークを疑ってください（SPEC §8）。公開されている競艇AIは概ね 80〜95% です。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.backtest --split valid \
    --payouts data/parquet/payouts.parquet

## 9. test 期間の最終評価 — **一度だけ**

`--final` を付けないと実行を拒否します。閾値を valid で決め切ってから、このセルを**1回だけ**回してください。何度も回して閾値を選び直したら、test はもう test ではありません。

In [ ]:
# !cd /content/kyotei-predict && python -m kyotei.backtest --split test --final \
#     --payouts data/parquet/payouts.parquet

## 10. 当日の買い目レポート

`reports/YYYY-MM-DD.md` に出力されます。Drive 上なのでそのまま残ります。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.predict \
    --payouts data/parquet/payouts.parquet --threshold 1.20

import datetime, pathlib
from IPython.display import Markdown, display

today = pathlib.Path(f'reports/{datetime.date.today():%Y-%m-%d}.md')
display(Markdown(today.read_text(encoding='utf-8') if today.exists() else '未生成'))

## 11. 実オッズのバックフィル（valid=2024）— ここが本番

日次ファイル(B/K)には**オッズが入っていない**。回収率の計算は実払戻だけで足りるが、
**買う前の期待値計算には全120通りの価格が必要**で、それが無いと EV は
`p × 定数` に退化し「このレースで市場が間違えている」を表現できない。

`odds3t` ページから**締切時オッズ120通り**が取れることは確認済み。
パースの正しさは独立に裏取りしてある（各オッズの逆数を払戻率75%で割り戻した
市場確率の合計が 1.0012）。

### 注意点

- **2024年は約53,000レース = 1req/s で約15時間。** Colab のセッション上限を超えるので、
  このセルを**何度も実行する**前提。取得済みはスキップされる。
- **レースは固定シードでシャッフルした順に回る。** 日付順だと途中で止まった時に
  「1〜4月だけ」＝季節バイアスのある使えないデータになる。シャッフルしておけば
  **どこで止まってもその時点までが2024年の無作為標本**として使える。
- 保存期間の制約から**2015〜2016のオッズは取れない**。だから学習期間ではなく
  valid(2024) を対象にしている。
- 先に 2024 年の番組表が必要（どのレースが存在するかを知るため）。


In [ ]:
# 2024年の番組表が無いと対象レースが分からないので先に取得（732ファイル、約13分）
!cd /content/kyotei-predict && python -m kyotei.download --start 2024-01-01 --end 2024-12-31 --kind both

In [ ]:
# まず 300 レースだけ試して、想定どおり動くか確かめる（約5分）
!cd /content/kyotei-predict && python -m kyotei.odds_backfill --year 2024 --limit 300

In [ ]:
# 本番。切れたらこのセルを再実行するだけでよい（取得済みはスキップ）
!cd /content/kyotei-predict && python -m kyotei.odds_backfill --year 2024

In [ ]:
# 取得状況の確認。月ごとに散っていれば、途中で止まっていても標本として使える
import pathlib
from kyotei import odds_backfill as ob

path = pathlib.Path('data/odds/odds3t_2024.jsonl')
frame = ob.to_frame(path)
races = frame.select(['race_date','stadium_id','race_no']).unique().height
print(f'取得済み: {races} レース ({frame.height} 行)')
print(ob.coverage(path))

## 12. 実オッズでバックテスト

`--odds` を渡すと、価格が「組番ごとの過去平均」ではなく**その日その レースの実オッズ**になる。
ここで初めて期待値ベースの購入判断が成立する。

**回収率が100%を超えたら、まずリークを疑う**（SPEC §8）。公開されている競艇AIは概ね80〜95%。


In [ ]:
!cd /content/kyotei-predict && python -m kyotei.backtest --split valid \
    --payouts data/parquet/payouts.parquet \
    --odds data/odds/odds3t_2024.parquet